# Robot Discovery & Tournament Analysis

This notebook performs EDA and training visualization for the robot endpoint estimation dataset.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error

from robot_positioning.config import EnvHelper
from robot_positioning.controller import TournamentManager
from robot_positioning.simulation import RunRecord, read_run_history, write_run_history

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


In [ ]:
repo_root = Path('.')
env_path = repo_root / '.env'
if not env_path.exists():
    env_path = repo_root / '.env.example'

env = EnvHelper(env_path)
manager = TournamentManager(env)

history_path = Path(env.get_val('RUN_HISTORY_PATH', str, default='run_history.csv'))
records = read_run_history(history_path)
if not records:
    generated = manager.physics_env.generate_runs(250, is_simulated=True) + manager.physics_env.generate_runs(80, is_simulated=False)
    write_run_history(history_path, generated)
    records = read_run_history(history_path)

df = pd.DataFrame([
    {
        'run_id': r.run_id,
        'total_tiles': r.total_tiles,
        'num_corners': r.num_corners,
        'start_battery_v': r.start_battery_v,
        'calibrations_total': r.calibrations_total,
        'actual_time_total': r.actual_time_total,
        'endpoint_error_cm': r.endpoint_error_cm,
        'endpoint_deviated_deg': r.endpoint_deviated_deg,
        'is_simulated': r.is_simulated,
    }
    for r in records
])

df['phase'] = np.where(df['is_simulated'], 'simulated', 'real')
weights = np.where(df['is_simulated'], env.get_val('SIM_WEIGHT', float, default=0.2), env.get_val('REAL_WEIGHT', float, default=1.0))

print(f'Rows: {len(df)} | Simulated: {df.is_simulated.sum()} | Real: {(~df.is_simulated).sum()}')


## 1) Data Profiling: Battery vs Total Time

In [ ]:
ax = sns.scatterplot(data=df, x='start_battery_v', y='actual_time_total', hue='phase', alpha=0.7)
sns.regplot(data=df, x='start_battery_v', y='actual_time_total', scatter=False, lowess=True, color='black', ax=ax)
ax.set_title('Battery Voltage vs Actual Time Total')
ax.set_xlabel('Start Battery (V)')
ax.set_ylabel('Actual Time Total (s)')
plt.show()


## 2) Error Analysis: Endpoint Error vs Calibrations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x='calibrations_total', y='endpoint_error_cm', hue='phase', alpha=0.7, ax=axes[0])
axes[0].set_title('Endpoint Error vs Calibrations')
axes[0].set_xlabel('Calibrations Total')
axes[0].set_ylabel('Endpoint Error (cm)')

plot_df = df.copy()
plot_df['abs_endpoint_error_cm'] = plot_df['endpoint_error_cm'].abs()
sns.lineplot(data=plot_df.groupby(['calibrations_total', 'phase'], as_index=False)['abs_endpoint_error_cm'].mean(),
             x='calibrations_total', y='abs_endpoint_error_cm', hue='phase', marker='o', ax=axes[1])
axes[1].set_title('Mean |Endpoint Error| by Calibration Count')
axes[1].set_ylabel('Mean Absolute Endpoint Error (cm)')

plt.tight_layout()
plt.show()


## 3) Tournament Leaderboard (6 Models)

In [ ]:
history = [
    RunRecord(
        run_id=int(row.run_id) if pd.notna(row.run_id) else None,
        total_tiles=int(row.total_tiles),
        num_corners=int(row.num_corners),
        start_battery_v=float(row.start_battery_v),
        calibrations_total=int(row.calibrations_total),
        actual_time_total=float(row.actual_time_total),
        endpoint_error_cm=float(row.endpoint_error_cm),
        endpoint_deviated_deg=float(row.endpoint_deviated_deg),
        is_simulated=bool(row.is_simulated),
    )
    for row in df.itertuples(index=False)
]

manager._reset_disqualifications()
manager.train_all(history, use_grid_search=False)
leaderboard = manager.export_leaderboard(history)
leaderboard_df = pd.DataFrame({'model': list(leaderboard.keys()), 'mae': list(leaderboard.values())})
leaderboard_df


In [ ]:
champion = leaderboard_df.sort_values('mae').iloc[0]['model']
colors = ['tab:red' if m == champion else 'tab:blue' for m in leaderboard_df['model']]

plt.figure(figsize=(12, 5))
plt.bar(leaderboard_df['model'], leaderboard_df['mae'], color=colors)
plt.xticks(rotation=30, ha='right')
plt.title(f'Tournament MAE Leaderboard (Champion: {champion})')
plt.ylabel('MAE (seconds)')
plt.tight_layout()
plt.show()


## 4) Champion Feature Importance

In [ ]:
feature_names = ['total_tiles', 'num_corners', 'start_battery_v', 'calibrations_total']
champion_competitor = manager.competitors[champion]
model = champion_competitor.estimator.model

if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
elif hasattr(model, 'coef_'):
    importances = np.abs(np.ravel(model.coef_))
else:
    importances = np.zeros(len(feature_names))

importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.title(f'Feature Importance - {champion}')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 5) Transfer Learning Curve (Simulated ➜ Real)

In [ ]:
real_records = [r for r in history if not r.is_simulated]
sim_records = [r for r in history if r.is_simulated]

if len(real_records) < 12:
    print('Not enough real records for a robust learning curve. Add more production runs and re-run.')
else:
    split_at = max(8, int(len(real_records) * 0.7))
    train_real = real_records[:split_at]
    holdout_real = real_records[split_at:]

    learning_x = []
    learning_y = []

    for k in range(1, len(train_real) + 1):
        train_subset = sim_records + train_real[:k]

        challenger = type(champion_competitor.estimator)(champion_competitor.estimator.param_grid)
        X_train = [r.features() for r in train_subset]
        y_train = [champion_competitor.strategy.training_target(r) for r in train_subset]
        w_train = [env.get_val('SIM_WEIGHT', float, default=0.2) if r.is_simulated else env.get_val('REAL_WEIGHT', float, default=1.0) for r in train_subset]
        challenger.train(X_train, y_train, w_train, use_grid_search=False)

        y_true = [r.actual_time_total for r in holdout_real]
        y_pred = []
        for r in holdout_real:
            raw = challenger.predict([r.features()])[0]
            y_pred.append(champion_competitor.strategy.prediction_to_time(r, raw))

        learning_x.append(k)
        learning_y.append(mean_absolute_error(y_true, y_pred))

    plt.figure(figsize=(10, 5))
    plt.plot(learning_x, learning_y, marker='o')
    plt.title(f'Transfer Learning Curve - {champion}')
    plt.xlabel('Number of Real-World Training Runs')
    plt.ylabel('Hold-out MAE (seconds)')
    plt.tight_layout()
    plt.show()


In [ ]:
manager.close()